# 01. Data Quality & Referential Integrity Engine

This notebook evaluates schema completeness, primary key uniqueness, foreign key consistency, and domain validity across the Instacart dataset using vectorized DuckDB queries.

In [ ]:
import duckdb
import json
from pathlib import Path

con = duckdb.connect()
pq_dir = Path('../data/parquet')
tables = ['orders', 'products', 'aisles', 'departments', 'order_products__prior', 'order_products__train']

for t in tables:
    con.execute(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM read_parquet('{(pq_dir / f'{t}.parquet').as_posix()}')")

print('DuckDB registered tables successfully!')

In [ ]:
# Completeness & Unexpected Nulls check
null_check = con.execute("SELECT count(*) FROM orders WHERE order_number > 1 AND days_since_prior_order IS NULL").fetchone()[0]
print(f'Unexpected nulls in orders (order_number > 1): {null_check}')

In [ ]:
# Foreign Key Integrity check
orphan_prods = con.execute("""
    SELECT count(*) 
    FROM order_products__prior opp 
    LEFT JOIN products p ON opp.product_id = p.product_id 
    WHERE p.product_id IS NULL
""").fetchone()[0]
print(f'Orphan products in prior items: {orphan_prods}')